A notebook to compute average measures of racial (minority) bias by state & chamber

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from knobs_functions import *
import warnings

warnings.filterwarnings('ignore')

Calculate the average value of "racial (minority) bias" metrics by state & chamber.

Note: Switch the list of ensembles for just the A0 table.

In [ ]:
from typing import List, Dict, Tuple, Any
from fetch import _score_mapping

metrics: List[str] = ["opportunity_districts", "coalition_districts", "proportional_opportunities", "proportional_coalitions", "mmd_black", "mmd_hispanic", "mmd_coalition"]
#, "mod_districts"] <<< Need to get this from Nick/Nithin
additions: Dict[str, str] = dict(zip(metrics, metrics))
_score_mapping.update(additions)
# ensembles = ["base0", "pop_minus", "pop_plus", "distpair", "ust", "distpair_ust", "reversible", "county25", "county50", "county75", "county100"]
ensembles = ["base0"]


averages: Dict[Tuple[str, str], Any] = dict()

for state, chamber in state_chamber_list:
    averages[(state, chamber)] = dict()
    for m in metrics:
        all_values: List[float] = []
        for e in ensembles:
            arr = fetch_score_array(state, chamber, e, m)
            all_values.extend(arr)
        mean_value = np.mean(all_values)
        averages[(state, chamber)][m] = mean_value

bias_table: Dict[Tuple[str, str], Any] = dict()

for combo, combo_averages in averages.items():
    bias_table[combo] = dict()

    pod = combo_averages["proportional_opportunities"]
    pcd = max(0, combo_averages["proportional_coalitions"] - pod)
    mmd_od = combo_averages["mmd_black"] + combo_averages["mmd_hispanic"]
    mmd_cd = max(0, combo_averages["mmd_coalition"] - mmd_od)
    dra_od = combo_averages["opportunity_districts"]
    dra_cd = max(0, combo_averages["coalition_districts"] - dra_od)

    bias_table[combo]["seats"] = num_seats_dict[combo]
    bias_table[combo]["pod"] = f"{pod:.0f}"
    bias_table[combo]["pcd"] = f"{pcd:.0f}"
    bias_table[combo]["mmd_od"] = f"{mmd_od:.4f}"
    bias_table[combo]["mmd_cd"] = f"{mmd_cd:.4f}"
    bias_table[combo]["dra_od"] = f"{dra_od:.4f}"
    bias_table[combo]["dra_cd"] = f"{dra_cd:.4f}"

# bias_table


In [6]:
headers = ["state", "chamber", "seats", "pod", "pcd", "mmd_od", "mmd_cd", "dra_od", "dra_cd"]
print(f"{','.join(headers)}")
for combo, _data in bias_table.items():
    state, chamber = combo
    print(f"{state},{chamber},{_data['seats']},{_data['pod']},{_data['pcd']},{_data['mmd_od']},{_data['mmd_cd']},{_data['dra_od']},{_data['dra_cd']}")

state,chamber,seats,pod,pcd,mmd_od,mmd_cd,dra_od,dra_cd
FL,congress,28,12,1,2.9762,0.1408,5.2500,11.6617
FL,upper,40,18,0,4.4116,0.0591,8.0551,16.4006
FL,lower,120,55,0,16.2767,0.0000,29.4722,42.1477
IL,congress,17,6,1,0.9146,0.6621,2.1515,6.8642
IL,upper,59,24,0,5.7614,0.0000,10.1391,18.0878
IL,lower,118,46,0,14.0210,0.0000,22.8799,32.1371
MI,congress,13,3,0,0.4289,0.0000,1.3279,1.1363
MI,upper,38,9,1,2.5922,0.0000,3.9562,3.2512
MI,lower,110,26,2,7.8853,0.0000,11.6828,12.1364
NC,congress,14,5,0,0.0000,0.0053,0.9868,7.4876
NC,upper,50,18,0,1.2078,0.4910,7.9995,17.7954
NC,lower,120,44,0,6.9454,0.0000,23.0778,34.9430
NY,congress,26,12,0,1.6782,2.0174,5.5760,9.8457
NY,upper,63,30,0,6.6273,0.2249,15.6570,21.1919
NY,lower,150,71,0,19.3592,0.0000,41.8964,44.5391
OH,congress,15,3,0,0.0000,0.0001,0.3137,1.6005
OH,upper,33,7,0,0.4731,0.0000,1.9819,3.1731
OH,lower,99,22,0,3.6589,0.0000,9.6281,7.9296
WI,congress,8,1,0,0.0000,0.0000,0.0293,0.8609
WI,upper,33,6,0,0.7259,0.0000,1.6969,2.0628
WI,lowe

Convert the dict to a pandas DataFrame and LaTex

TODO's
* Need to generate the LaTex columns as right-justified
* Except the header's which should be centered

I tweaked both by hand

In [3]:
def make_partisan_bias_table(*, latex_filename = None, rounding: int = 2):
    """ This is modeled after mean_diff_table() """

    index_list = [f'{a[0]} {a[1]}' for a in state_chamber_list]
    df = pd.DataFrame(columns = metrics, index = index_list)

    for state, chamber in state_chamber_list:
        for m in metrics:
            multiplier = 1 if m == "declination" else 100
            df.loc[f'{state} {chamber}', m] = bias_table[(state, chamber)][m] * multiplier
    df = df.applymap(pd.to_numeric)
    df = df.round(rounding)
    df_latex = df.copy()
    df_latex = df_latex.applymap(lambda x: f"{x:.2f}") # round values

    # combine the values and markings into dataframes to return and for Latex
    state_chamber_size_dict = {f'{state} {chamber}': f'{state} {num_seats_dict[(state, chamber)]}' 
                            for state, chamber in state_chamber_list}
    for state, chamber in state_chamber_list:
        for m in metrics:
            val = df.loc[f'{state} {chamber}', m]
            df_latex.loc[f'{state} {chamber}', m] = f'\\textcolor{{black}}{{ {val:.2f} }}' # TODO

    greek = {
        'alpha': 'α',
        'beta': 'β', 
        'delta': 'δ'
    }

    metrics_name_dict: Dict[str, str] = {
        "disproportionality": "PR", 
        "efficiency_gap": "EG",
        "geometric_seats_bias": greek['beta'],
        "seats_bias": greek['alpha'] + "_s",
        "votes_bias": greek['alpha'] + "_v",
        "mean_median_average_district": "mM", 
        "lopsided_outcomes": "LO", 
        "declination": greek['delta']
        }

    if latex_filename is not None:
        df_latex.rename(columns=metrics_name_dict, index=state_chamber_size_dict, inplace=True)
        df_latex.to_latex(latex_filename, escape=False)

    return df

Note: Switch the output location for the LaTeX for the A0 table

In [ ]:
# df = make_partisan_bias_table(latex_filename='latex tables/partisan_bias_table.tex')
df = make_partisan_bias_table(latex_filename='latex tables/partisan_bias_table_A0.tex')
df

,disproportionality,efficiency_gap,geometric_seats_bias,seats_bias,votes_bias,mean_median_average_district,lopsided_outcomes,declination
FL congress,5.10,3.47,1.83,2.23,0.70,2.23,0.53,6.01
FL upper,3.57,1.93,0.27,0.54,0.17,1.26,0.18,4.25
FL lower,3.90,2.26,1.42,1.69,0.65,2.03,1.15,6.92
IL congress,-7.13,1.04,3.64,5.63,1.99,2.91,10.67,6.88
IL upper,-7.97,0.20,3.04,5.55,1.94,3.17,9.97,3.02
IL lower,-7.84,0.33,2.73,4.21,1.54,3.03,9.83,1.70
MI congress,3.63,5.52,8.02,8.34,2.27,3.64,5.12,11.84
MI upper,4.36,6.24,7.10,7.15,2.71,3.95,6.03,13.68
MI lower,4.36,6.24,6.71,6.73,2.82,4.27,6.79,14.67
NC congress,4.70,4.13,3.15,3.19,0.78,0.98,0.42,4.32
